# Day 4: Launch the Fine-Tune

**Goal:** Submit a real SageMaker training job. Walk away for 2–4 hours. Come back to a fine-tuned Mistral-7B.

**Cost:** ~$20–25 for a 3-epoch run on `ml.g5.12xlarge`.

**Prereqs (if you skip these, things break):**
- Day 2 dataset is in S3 (`training_input_path`)
- Quota for `ml.g5.12xlarge for training job usage` approved (Service Quotas → SageMaker)
- `scripts/run_clm.py` and `scripts/requirements.txt` exist in this folder
- Hugging Face token in an env var (below)

## Step 1: Check the environment

In [1]:
import os

# scripts/ folder must exist with run_clm.py and requirements.txt
assert os.path.exists("./scripts/run_clm.py"), "Missing scripts/run_clm.py"
assert os.path.exists("./scripts/requirements.txt"), "Missing scripts/requirements.txt"
print("✅ Training script and requirements.txt found")

✅ Training script and requirements.txt found


## Step 2: Set your Hugging Face token

This is passed to the training container so it can download Mistral from the Hub.

In [ ]:
HF_TOKEN = "hf_[REPLACE WITH YOURS]]"   # <- paste your HF token here
assert HF_TOKEN.startswith("hf_"), "Token must start with 'hf_'"
print("✅ HF token set")

✅ HF token set


## Step 3: SageMaker session

In [3]:
import time
import sagemaker
from sagemaker.huggingface import HuggingFace

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Region: {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role:   arn:aws:iam::743220303713:role/SageMakerExecutionRole-Manual
Region: us-east-2


## Step 4: Paths from Day 2

Replace the bucket name with yours.

In [ ]:
bucket_name         = "llm-bootcamp-data-XX-2026"                       # <- your bucket
training_input_path = f"s3://{bucket_name}/processed/mistral/dolly/train"
model_id            = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Training data: {training_input_path}")
print(f"Base model:    {model_id}")

Training data: s3://llm-bootcamp-data-ts-2026/processed/mistral/dolly/train
Base model:    mistralai/Mistral-7B-Instruct-v0.3


## Step 5: Hyperparameters

These get passed as CLI args to `run_clm.py` inside the container.

- `per_device_train_batch_size=4` × 4 GPUs = effective batch size 16
- `lr=2e-4` — standard QLoRA learning rate (higher than full FT because only 0.09% of params train)
- `epochs=3` — sweet spot for Dolly. Going to 5+ risks overfitting.

In [5]:
hyperparameters = {
    "model_id": model_id,
    "dataset_path": "/opt/ml/input/data/training",   # SageMaker mount point — don't change
    "epochs": 3,
    "lr": 2e-4,
    "per_device_train_batch_size": 4,
    "bf16": True,
    "gradient_checkpointing": True,
    "merge_weights": True,
}

for k, v in hyperparameters.items():
    print(f"  {k}: {v}")

  model_id: mistralai/Mistral-7B-Instruct-v0.3
  dataset_path: /opt/ml/input/data/training
  epochs: 3
  lr: 0.0002
  per_device_train_batch_size: 4
  bf16: True
  gradient_checkpointing: True
  merge_weights: True


## Step 6: Build the HuggingFace estimator

⚠️ **Critical:** `HUGGINGFACE_HUB_CACHE` must be spelled exactly right. A typo here = cryptic disk-space errors 20 min into training.

⚠️ `volume_size=300` GB is chosen deliberately. The model is 15 GB, but checkpoints + merged output + temp files need 10–15× that headroom.

In [6]:
job_name = f"mistral-qlora-{time.strftime('%Y-%m-%d-%H-%M-%S', time.localtime())}"

estimator = HuggingFace(
    entry_point="run_clm.py",
    source_dir="./scripts",
    base_job_name=job_name,
    instance_type="ml.g5.12xlarge",   # 4× A10G, 96 GB total VRAM, ~$5–7/hr
    instance_count=1,
    role=role,
    volume_size=300,                  # GB of EBS
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    hyperparameters=hyperparameters,
    environment={
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache",         # MUST be exact spelling
        "HF_TOKEN": HF_TOKEN,
    },
)

print(f"Estimator ready. Job name: {job_name}")
print("No charges yet — nothing runs until .fit() is called.")

Estimator ready. Job name: mistral-qlora-2026-04-23-03-10-46
No charges yet — nothing runs until .fit() is called.


## Step 7: 🔥 LAUNCH THE TRAINING JOB

⚠️ **This is the cell that starts costing money.** Expect ~$20–25 total for a 3-epoch run.

Once submitted:
1. Logs stream into this cell for ~20 minutes (download + setup)
2. Training begins when you see `{"loss": ...}` lines
3. You can interrupt this cell (Kernel → Interrupt) **without stopping the job** — the job runs independently on SageMaker
4. Monitor progress in AWS Console → SageMaker AI → Training jobs

In [ ]:
estimator.fit({"training": training_input_path}, wait=True)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: mistral-qlora-2026-04-23-03-10-46-2026-04-23-03-10-54-639


2026-04-23 03:10:56 Starting - Starting the training job......
2026-04-23 03:11:53 Downloading - Downloading input data...
2026-04-23 03:12:03 Downloading - Downloading the training image...............
2026-04-23 03:15:00 Training - Training image download completed. Training in progress......bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
2026-04-23 03:15:40,621 sagemaker-training-toolkit INFO     Impor

## Step 8: Find the trained model

Once the job completes, the merged model lives at the path below, in ~15 shards totaling about 15 GB.

In [ ]:
model_s3_uri = estimator.model_data
print("=" * 60)
print("SAVE THIS FOR DAY 5:")
print("=" * 60)
print(f"model_s3_uri = '{model_s3_uri}'")
print(f"region       = '{region}'")
print(f"role         = '{role}'")
print("=" * 60)

## 🚨 Shutdown ritual

1. Training job runs to completion on its own — you don't need this notebook open
2. You CAN safely close JupyterLab and stop the space once the job shows `InProgress` in the console
3. When the job finishes: check **SageMaker → Training jobs** for `Completed` status
4. No endpoint is running yet. No ongoing GPU charges after `Completed`.

---

✅ **Day 4 complete.** See you on Day 5 to ship it.